# BYOT (Bring Your Own Target)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brightbandtech/extremeweatherbench/blob/main/notebooks/your_own_target.ipynb)

This notebook demonstrates how to subclass `TargetBase` to bring your own observation target into EWB. Uses a custom GHCNh wrapper as an example. Optimized for Google Colab with a small demo case.

In [ ]:
!pip install -q extremeweatherbench

In [ ]:
import datetime
import extremeweatherbench as ewb
from extremeweatherbench.cases import IndividualCase
from extremeweatherbench.regions import BoundingBoxRegion

# Mini-case: 2018 Japan Heat Wave — Colab-optimized
demo_case = IndividualCase(
    case_id_number=9003,
    title="2018 Japan Heat Wave (demo)",
    start_date=datetime.datetime(2018, 7, 21),
    end_date=datetime.datetime(2018, 7, 24),
    location=BoundingBoxRegion.create_region(
        latitude_min=33.0,
        latitude_max=39.0,
        longitude_min=133.0,
        longitude_max=139.0,
    ),
    event_type="heat_wave",
)
cases = [demo_case]

In [ ]:
import dataclasses
import pandas as pd
import polars as pl
import xarray as xr
from extremeweatherbench import inputs, utils


@dataclasses.dataclass
class CustomGHCN(inputs.TargetBase):
    """GHCNh parquet via TargetBase, with explicit unit handling."""

    name: str = "CustomGHCN"
    source: str = ewb.DEFAULT_GHCN_URI

    def _open_data_from_source(self):
        return pl.scan_parquet(
            self.source,
            storage_options={"anon": True},
        )

    def subset_data_to_case(self, data, case_metadata, **kwargs):
        bounds = case_metadata.location.as_geopandas().total_bounds
        time_min = case_metadata.start_date - pd.Timedelta(days=1)
        time_max = case_metadata.end_date + pd.Timedelta(days=1)
        return data.filter(
            (pl.col("valid_time") >= time_min)
            & (pl.col("valid_time") <= time_max)
            & (pl.col("latitude") >= bounds[1])
            & (pl.col("latitude") <= bounds[3])
            & (pl.col("longitude") >= bounds[0])
            & (pl.col("longitude") <= bounds[2])
        )

    def _custom_convert_to_dataset(self, data):
        df = data.collect().to_pandas()
        df["surface_air_temperature"] += 273.15
        df["longitude"] = utils.convert_longitude_to_360(
            df["longitude"]
        )
        df = df.set_index(
            ["valid_time", "latitude", "longitude"]
        )
        return xr.Dataset.from_dataframe(
            df[~df.index.duplicated(keep="first")], sparse=True
        )

    def maybe_align_forecast_to_target(
        self, forecast_data, target_data
    ):
        return inputs.align_forecast_to_target(
            forecast_data, target_data
        )

In [ ]:
custom_target = CustomGHCN()

forecast = ewb.ZarrForecast(
    source="gs://weatherbench2/datasets/hres/2016-2022-0012-1440x721.zarr",
    name="HRES",
    variable_mapping=ewb.HRES_metadata_variable_mapping,
    storage_options={"remote_options": {"anon": True}},
)

eval_objects = [
    ewb.EvaluationObject(
        event_type="heat_wave",
        metric_list=[
            ewb.metrics.MeanAbsoluteError(
                forecast_variable="surface_air_temperature",
                target_variable="surface_air_temperature",
            ),
        ],
        target=custom_target,
        forecast=forecast,
    ),
]

runner = ewb.evaluation(
    case_metadata=cases,
    evaluation_objects=eval_objects,
)
outputs = runner.run_evaluation()
print(outputs)